# P1-16b · External test sets for Paper 1: DMID (proxy rule) and mini-MIAS (expert masks) (CPU)
**Runtime: CPU** (about 30–60 min; resumable). Downloads the public DMID full-field digital mammograms (figshare 24522883)
to the Colab VM, converts each to 8-bit grey values, applies exactly the Paper 1 labelling rule
(`qseg.data_prep.generate_breast_mask`: Otsu, opening + closing with an elliptical element of 20 px at the Paper 1 image height of 3,584 px, scaled to each image's height, largest contour) and the Paper 1
preprocessing (`resize_pair`: pad away from the chest wall, 256 × 256). Exact duplicate images (MD5 of the grey image) are removed.
Writes only to `MyDrive/P1_v2/dmid_ext/` (images/, masks/, manifest.csv, qc.png). QSEG_FINAL is read only (code import).


In [ ]:
# 1 · Drive, paths
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, io, json, hashlib, subprocess, zipfile
import numpy as np, pandas as pd, requests, cv2
ROOT = '/content/drive/MyDrive/QSEG_FINAL'            # read only: qseg code
EXT  = '/content/drive/MyDrive/P1_v2/dmid_ext'
for d in (f'{EXT}/images', f'{EXT}/masks'): os.makedirs(d, exist_ok=True)
assert not EXT.startswith(ROOT)
sys.path.insert(0, ROOT + '/code')
from qseg.data_prep import generate_breast_mask, resize_pair
!pip -q install tifffile imagecodecs


In [ ]:
# 2 · Download the DMID TIFF images (figshare) to the VM
import tifffile
from PIL import Image
RAW, LOCAL = '/content/dmid_raw', '/content/dmid'
os.makedirs(RAW, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
art = requests.get('https://api.figshare.com/v2/articles/24522883', timeout=60).json()
for f in art['files']:
    if 'TIFF' not in f['name']: continue
    dst = f"{RAW}/{f['name']}"
    if not os.path.exists(dst) or os.path.getsize(dst) != f['size']:
        with requests.get(f['download_url'], stream=True, timeout=900) as r:
            r.raise_for_status()
            with open(dst, 'wb') as o:
                for ch in r.iter_content(1 << 20): o.write(ch)
    if dst.endswith('.zip'):
        subprocess.run(['unzip', '-q', '-o', dst, '-d', LOCAL], check=False)
IMG_DIR = glob.glob(f'{LOCAL}/**/TIFF Images', recursive=True)[0]
files = sorted(glob.glob(f'{IMG_DIR}/*.tif'))
print(len(files), 'DMID images')


In [ ]:
# 3 · Grey conversion, proxy mask (Paper 1 rule), Paper 1 preprocessing
def read_tif(p):
    try: a = tifffile.imread(p)
    except Exception: a = np.array(Image.open(p))
    a = np.asarray(a)
    if a.ndim == 3 and a.shape[0] in (1, 3, 4) and a.shape[-1] not in (1, 3, 4): a = np.moveaxis(a, 0, -1)
    if a.ndim == 3: a = a[..., :3].mean(-1) if a.shape[-1] >= 3 else a[..., 0]
    return a
def to_uint8(img):
    img = img.astype(np.float32); nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    return (np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
rows, seen = [], {}
for p in files:
    fid = os.path.basename(p)[:-4].upper()
    gray = to_uint8(read_tif(p))
    h = hashlib.md5(gray.tobytes()).hexdigest()
    if h in seen:
        rows.append(dict(file_id=fid, duplicate_of=seen[h])); continue
    seen[h] = fid
    k = max(3, round(20 * gray.shape[0] / 3584))            # Paper 1 kernel (20 px at 3584 px height) scaled to the image height
    mask, thr, _ = generate_breast_mask(gray, k)
    img_s, msk_s = resize_pair(gray, mask, 256, True)
    cv2.imwrite(f'{EXT}/images/{fid}.png', img_s); cv2.imwrite(f'{EXT}/masks/{fid}.png', msk_s)
    rows.append(dict(file_id=fid, duplicate_of='', patient_id=fid, orig_height=gray.shape[0], orig_width=gray.shape[1],
                     otsu_threshold=thr, mask_frac=round(float((msk_s > 127).mean()), 4)))
man = pd.DataFrame(rows); man.to_csv(f'{EXT}/manifest.csv', index=False)
u = man[man.duplicate_of == '']
print('images:', len(man), '| unique:', len(u), '| duplicates removed:', int((man.duplicate_of != '').sum()))
print(u[['orig_height', 'orig_width', 'mask_frac']].describe().round(3).to_string())


In [ ]:
# 4 · QC grid (6 random images with mask outlines)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 6, figsize=(15, 3))
for a, fid in zip(ax, u.file_id.sample(6, random_state=0)):
    im = cv2.imread(f'{EXT}/images/{fid}.png', 0); mk = cv2.imread(f'{EXT}/masks/{fid}.png', 0)
    rgb = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB); cs, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(rgb, cs, -1, (255, 60, 60), 2); a.imshow(rgb); a.set_title(fid, fontsize=8); a.axis('off')
plt.tight_layout(); plt.savefig(f'{EXT}/qc.png', dpi=110); plt.show()
print('done; files in', EXT)


## External set 2: mini-MIAS (digitized screen-film) with expert-corrected breast and pectoral masks
Images: mini-MIAS (322 films, 1024 × 1024, 8-bit). Masks: Verboom et al., Zenodo 10.5281/zenodo.10149914 (CC BY 4.0):
background 0, breast 1, pectoral muscle 2, drawn by medical imaging scientists (Otsu start, manual correction).
The Paper 1 rule is applied to every image as well, so (i) the agreement of the proxy rule with the expert masks is measured
and (ii) the trained models can later be scored against the expert masks. Writes to `MyDrive/P1_v2/mias_ext/`.

In [ ]:
# 5 · Download mini-MIAS images and the expert masks to the VM
!pip -q install pynrrd
import nrrd, tarfile
MIAS = '/content/drive/MyDrive/P1_v2/mias_ext'
for d in (f'{MIAS}/images', f'{MIAS}/masks_expert', f'{MIAS}/masks_expert_breast', f'{MIAS}/masks_proxy'): os.makedirs(d, exist_ok=True)
MR = '/content/mias_raw'; os.makedirs(MR, exist_ok=True)
ok = False
for url in ('http://peipa.essex.ac.uk/pix/mias/all-mias.tar.gz', 'https://peipa.essex.ac.uk/pix/mias/all-mias.tar.gz'):
    try:
        r = requests.get(url, timeout=600); r.raise_for_status()
        open(f'{MR}/all-mias.tar.gz', 'wb').write(r.content)
        tarfile.open(f'{MR}/all-mias.tar.gz').extractall(MR); ok = True; break
    except Exception as e:
        print('failed', url, e)
assert ok, 'mini-MIAS download failed: send a screenshot of this error'
pgms = sorted(glob.glob(f'{MR}/**/mdb*.pgm', recursive=True)); print(len(pgms), 'mini-MIAS images')
rec = requests.get('https://zenodo.org/api/records/10149914', timeout=60).json()
for f in rec['files']:
    url = f['links'].get('self') or f['links'].get('download')
    dst = f"{MR}/{f['key']}"
    open(dst, 'wb').write(requests.get(url, timeout=600).content)
    if dst.endswith('.zip'): zipfile.ZipFile(dst).extractall(f'{MR}/masks')
nrrds = sorted(glob.glob(f'{MR}/masks/**/*.nrrd', recursive=True)); print(len(nrrds), 'expert mask files')


In [ ]:
# 6 · Align expert masks, apply the Paper 1 rule, Paper 1 preprocessing, rule-vs-expert agreement
import cv2
def dice_np(a, b):
    a, b = a > 0, b > 0; s = a.sum() + b.sum()
    return 1.0 if s == 0 else 2 * (a & b).sum() / s
import re
mask_by_id = {}
for p in nrrds:
    m = re.search(r'mdb(\d{3})', os.path.basename(p).lower())
    if m: mask_by_id['mdb' + m.group(1)] = p
rows = []
for p in pgms:
    fid = os.path.basename(p)[:-4].lower()
    if fid not in mask_by_id: rows.append(dict(file_id=fid, status='no expert mask')); continue
    gray = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    lab, _ = nrrd.read(mask_by_id[fid]); lab = np.squeeze(lab)
    k = max(3, round(20 * gray.shape[0] / 3584))            # Paper 1 kernel (20 px at 3584 px height) scaled to the image height
    proxy, thr, _ = generate_breast_mask(gray, k)
    cands = {'id': lab, 'T': lab.T, 'ud': lab[::-1], 'Tud': lab.T[::-1], 'lr': lab[:, ::-1], 'Tlr': lab.T[:, ::-1]}
    cands = {k: v for k, v in cands.items() if v.shape == gray.shape}
    best = max(cands, key=lambda k: dice_np(cands[k] > 0, proxy))
    lab = np.ascontiguousarray(cands[best])
    full = ((lab == 1) | (lab == 2)).astype(np.uint8) * 255; breast = (lab == 1).astype(np.uint8) * 255
    side = 'left' if (proxy[:, gray.shape[1] // 2:] > 0).sum() >= (proxy[:, :gray.shape[1] // 2] > 0).sum() else 'right'
    from qseg.data_prep import pad_to_square
    def prep(a, interp):
        return cv2.resize(pad_to_square(a, side), (256, 256), interpolation=interp)
    img_s = prep(gray, cv2.INTER_AREA)
    ms = {k: ((prep(v, cv2.INTER_NEAREST) > 127) * 255).astype(np.uint8) for k, v in (('proxy', proxy), ('full', full), ('breast', breast))}
    cv2.imwrite(f'{MIAS}/images/{fid}.png', img_s); cv2.imwrite(f'{MIAS}/masks_proxy/{fid}.png', ms['proxy'])
    cv2.imwrite(f'{MIAS}/masks_expert/{fid}.png', ms['full']); cv2.imwrite(f'{MIAS}/masks_expert_breast/{fid}.png', ms['breast'])
    rows.append(dict(file_id=fid, status='ok', orientation=best, has_pectoral=bool((lab == 2).any()),
                     dice_rule_vs_expert_full=round(dice_np(ms['proxy'], ms['full']), 4),
                     dice_rule_vs_expert_breast=round(dice_np(ms['proxy'], ms['breast']), 4),
                     mask_frac_expert=round(float((ms['full'] > 0).mean()), 4)))
mm = pd.DataFrame(rows); mm.to_csv(f'{MIAS}/manifest.csv', index=False)
ok = mm[mm.status == 'ok']
print('images:', len(mm), '| with expert mask:', len(ok), '| orientation used:', ok.orientation.value_counts().to_dict())
print('Paper 1 rule vs expert masks (per-image Dice), breast + pectoral:', ok.dice_rule_vs_expert_full.describe().round(3).to_dict())
print('Paper 1 rule vs expert masks (per-image Dice), breast only     :', ok.dice_rule_vs_expert_breast.describe().round(3).to_dict())


In [ ]:
# 7 · QC grid: image, expert mask (green), proxy rule (red)
fig, ax = plt.subplots(1, 6, figsize=(15, 3))
for a, fid in zip(ax, ok.file_id.sample(6, random_state=0)):
    im = cv2.imread(f'{MIAS}/images/{fid}.png', 0); rgb = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB)
    for sub, col in (('masks_expert', (60, 220, 60)), ('masks_proxy', (255, 60, 60))):
        mk = cv2.imread(f'{MIAS}/{sub}/{fid}.png', 0); cs, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(rgb, cs, -1, col, 2)
    a.imshow(rgb); a.set_title(fid, fontsize=8); a.axis('off')
plt.tight_layout(); plt.savefig(f'{MIAS}/qc.png', dpi=110); plt.show()
print('done; files in', MIAS)
